# Genie API - 設定情報の管理

このノートブックは、Genie APIを用いて指定したGenie Spaceの設定を管理します。

**処理フロー:**
1. Genie Spaceの設定を取得(JSON)
2. Genie Spaceに設定を反映

In [0]:
%run ./00_config

In [0]:
%run ./_init_genie_api

## 1. genie設定情報を取得する
[Get Genie Space](https://docs.databricks.com/api/workspace/genie/getspace#serialized_space)<br>
Genie APIは
- `GET /api/2.0/genie/spaces/{space_id}?include_serialized_space=true` でスペース構成のシリアライズを取得できます
- `POST /api/2.0/genie/spaces` および `PATCH /api/2.0/genie/spaces/{space_id}` で適用できます

In [0]:
# APIからデータを取得
url = f"{GENIE_API_BASE}?include_serialized_space=true"
resp = requests.get(url, headers=HEADERS)
full_data = resp.json()

# 何という項目名（キー）が存在するかだけを表示
print(f"取得できた項目一覧: {list(full_data.keys())}")

# もし 'space' という項目があれば、その中身のキーも表示
if "space" in full_data:
    print(f"spaceの中にある項目一覧: {list(full_data['space'].keys())}")

In [0]:
# APIからデータを取得
url = f"{GENIE_API_BASE}?include_serialized_space=true"
resp = requests.get(url, headers=HEADERS)
full_data = resp.json()

# キーが 'title' であることが分かったので、こちらを参照します
space_name = full_data.get("title", "名前が取得できませんでした")

# 設定の中身（シリアライズされた部分）
config_str = full_data.get("serialized_space", "{}")
config = json.loads(config_str) if isinstance(config_str, str) else config_str

print(f"=== [Genie スペース名] ===")
print(space_name)

print("\n--- [Genie 指示書 (Instructions)] ---")
for inst in config.get("instructions", {}).get("text_instructions", []):
    print("".join(inst.get("content", [])))

print("\n--- [紐づけテーブル (Tables)] ---")
for table in config.get("data_sources", {}).get("tables", []):
    print(f"- {table.get('identifier')}")

print("\n--- [結合条件 (Joins)] ---")
for join in config.get("instructions", {}).get("join_specs", []):
    print(f"- {' '.join(join.get('sql', []))}")

## 2. Genie に追加設定する

In [0]:
'''
最新の設定を取得
'''
config_str = get_genie_space_config()
config = json.loads(config_str) if isinstance(config_str, str) else config_str

'''
設定を書き換える
'''
# 指示の追加
if "instructions" in config and "text_instructions" in config["instructions"]:
    # すでに同じ指示がなければ追加
    if "\n* テスト追加" not in config["instructions"]["text_instructions"][0]["content"]:
        config["instructions"]["text_instructions"][0]["content"].append("\n* テスト追加")

# テーブルの追加
new_table_id = f"{catalog}.{schema}.feedbacks"

# 既存のテーブル一覧を取得
current_tables = config["data_sources"]["tables"]

# 重複チェック（すでになければ追加）
if not any(t.get('identifier') == new_table_id for t in current_tables):
    current_tables.append({"identifier": new_table_id})

# エラー回避のため、テーブル名をアルファベット順にソートする
config["data_sources"]["tables"] = sorted(current_tables, key=lambda x: x['identifier'])

# 更新
url = GENIE_API_BASE
payload = {
    "serialized_space": json.dumps(config)
}

try:
    resp = requests.patch(url, headers=HEADERS, data=json.dumps(payload))
    resp.raise_for_status()
    print("成功：ソートして適用しました！")
except requests.exceptions.HTTPError as e:
    print(f"エラー: {e}")
    print(f"詳細: {e.response.text}")